# 03 · Model development and evaluation

Every model is trained on the **training** part, compared on the **validation** part, and only the three best primary classifiers are scored on the **test** part, once, at the end.

Order of work:
1. Shared helpers: one threshold rule and one evaluation function used by every model
2. Baseline "always normal"
3. PCA reconstruction-error detector (learns normal blocks only)
4. Logistic regression, with and without class weights
5. Decision tree: depth sweep for bias-variance, then readable rules
6. kNN, choosing k on validation
7. Gaussian Naive Bayes
8. Linear SVM
9. Random forest
10. AdaBoost
11. Small MLP
12. Validation comparison → test the top three once → save the best model

**It creates** `reports/tables/model_comparison.csv`, `final_test_results.csv`, `model_settings.csv`, `event_importance.csv`, `decision_tree_rules.txt`, `models/best_final_model.pkl` and the figures `tree_depth_bias_variance.png`, `decision_tree.png`, `validation_precision_recall.png`, `test_roc_curves.png`.

In [1]:
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, log_loss,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

root = Path.cwd()
if not (root / 'datasets').exists():
    root = root.parent
processed = root / 'datasets' / 'processed'
figures = root / 'reports' / 'figures'
tables = root / 'reports' / 'tables'
models_folder = root / 'models'
models_folder.mkdir(exist_ok=True)

NORMAL_COLOUR = '#8a8a8a'
ANOMALY_COLOUR = '#d4591c'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})
print('scikit-learn version:', sklearn.__version__)

scikit-learn version: 1.9.1


In [2]:
df = pd.read_csv(processed / 'hdfs_clean.csv')
splits = np.load(processed / 'splits.npz')
event_columns = [name for name in df.columns if name.startswith('E') and name[1:].isdigit() and name != 'E22']
assert df['E22'].iloc[splits['train_idx']].nunique() == 1, 'E22 is not constant in training'

X = df[event_columns].values
y = df['target'].values
X_train, y_train = X[splits['train_idx']], y[splits['train_idx']]
X_val, y_val = X[splits['validation_idx']], y[splits['validation_idx']]
X_test, y_test = X[splits['test_idx']], y[splits['test_idx']]

for part_name, part_y in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    print(f'{part_name:10s} blocks {len(part_y):7d}   anomalies {part_y.sum():6d}   rate {part_y.mean():.6f}')

train      blocks  345035   anomalies  10102   rate 0.029278
validation blocks  115013   anomalies   3368   rate 0.029284
test       blocks  115013   anomalies   3368   rate 0.029284


## Alarm threshold

A score at or above the threshold raises an alarm. Thresholds are selected using validation data only, minimising `10 × false negatives + false positives`. The 10:1 ratio is a project assumption. Every distinct validation score and the never-alarm decision are examined; ties use the lowest threshold.


In [3]:
FN_COST = 10  # project assumption: a missed anomaly costs ten false alarms
FP_COST = 1

def predict_at(scores, threshold):
    return (scores >= threshold).astype(int)

def choose_threshold(y_true, scores):
    # Sort once, then evaluate every distinct score with cumulative counts.
    order = np.argsort(scores, kind='stable')
    s, y = scores[order], y_true[order]
    starts = np.r_[0, np.flatnonzero(np.diff(s)) + 1]
    positive_before = np.r_[0, np.cumsum(y)][starts]
    negative_before = starts - positive_before
    costs = FN_COST * positive_before + FP_COST * ((len(y) - y.sum()) - negative_before)
    candidates = np.r_[s[starts], np.inf]
    costs = np.r_[costs, FN_COST * y.sum()]
    best = int(np.argmin(costs))
    return candidates[best], costs[best]


## Evaluation helpers

`evaluate` turns scores and a threshold into every metric on the syllabus. AUC uses the raw scores (it measures ranking, so it needs no threshold).
Log loss is only defined for scores that are probabilities; for the PCA detector and the SVM it is left empty.

`record` runs a fitted model on the validation set, chooses its threshold, prints the metrics and confusion matrix, and stores the model, its scores and its settings for later.

In [4]:
def evaluate(y_true, scores, threshold, gives_probability):
    predicted = predict_at(scores, threshold)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    row = {
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, predicted),
        'precision': precision_score(y_true, predicted, zero_division=0),
        'recall': recall_score(y_true, predicted),
        'f1': f1_score(y_true, predicted, zero_division=0),
        'auc': roc_auc_score(y_true, scores),
        'log_loss': np.nan,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'cost': FN_COST * fn + FP_COST * fp,
    }
    if gives_probability:
        row['log_loss'] = log_loss(y_true, scores, labels=[0, 1])
    return row


def show(row):
    print(f"threshold {row['threshold']:.6g}   cost {row['cost']}")
    print(f"accuracy {row['accuracy']:.6f}   precision {row['precision']:.6f}   recall {row['recall']:.6f}")
    print(f"F1 {row['f1']:.6f}   AUC {row['auc']:.6f}   log loss {row['log_loss']:.6f}")
    matrix = pd.DataFrame([[row['tn'], row['fp']], [row['fn'], row['tp']]],
                          index=['actual normal', 'actual anomaly'],
                          columns=['predicted normal', 'predicted anomaly'])
    display(matrix)

In [5]:
results = []            # one row of validation metrics per model
settings = []           # one row of hyperparameters per model
fitted = {}             # model name -> (model, score function, gives probability)
validation_scores = {}  # model name -> validation scores, reused for plots


def record(name, model, score_function, gives_probability, setting_text, training_rows):
    scores = score_function(model, X_val)
    threshold, cost = choose_threshold(y_val, scores)
    row = evaluate(y_val, scores, threshold, gives_probability)
    row['model'] = name
    results.append(row)
    settings.append({'model': name, 'settings': setting_text, 'training_rows': training_rows,
                     'scikit_learn_version': sklearn.__version__})
    fitted[name] = (model, score_function, gives_probability)
    validation_scores[name] = scores
    show(row)

Three kinds of score are used: a predicted probability of anomaly, the signed distance from the SVM boundary, and the PCA reconstruction error.

In [6]:
def anomaly_probability(model, X_part):
    return model.predict_proba(X_part)[:, 1]


def svm_margin(model, X_part):
    return model.decision_function(X_part)


def reconstruction_error(model, X_part):
    # Project onto the normal-behaviour components and back; what is lost is the error.
    scaled = model.named_steps['standardscaler'].transform(X_part)
    pca = model.named_steps['pca']
    rebuilt = pca.inverse_transform(pca.transform(scaled))
    return ((scaled - rebuilt) ** 2).sum(axis=1)

def knn_probability(model, X_part):
    # Score identical rows once; restore a result for every original block.
    unique_rows, inverse = np.unique(X_part, axis=0, return_inverse=True)
    return model.predict_proba(unique_rows)[:, 1][inverse]


## 2 · Baseline: always "normal"

`DummyClassifier(strategy='prior')` gives every block the same score: the training anomaly rate. With one score for everyone, the cheapest threshold is "never alarm". This is the number every real model must beat: high accuracy, zero recall.

In [7]:
baseline = DummyClassifier(strategy='prior')
baseline.fit(X_train, y_train)
record('Baseline (always normal)', baseline, anomaly_probability, True,
       "DummyClassifier(strategy='prior')", len(y_train))

threshold inf   cost 33680
accuracy 0.970716   precision 0.000000   recall 0.000000
F1 0.000000   AUC 0.500000   log loss 0.132243


,predicted normal,predicted anomaly
actual normal,111645,0
actual anomaly,3368,0


## 3 · PCA reconstruction-error detector

This detector never sees an anomaly while learning. It is fitted on **normal training blocks only**:
1. standardise the counts;
2. keep the PCA components that explain 95% of the variance of normal blocks;
3. score a block by how badly those components rebuild it (sum of squared errors).

A normal-looking block is rebuilt well; an unusual one is not. The labels are used only to choose the alarm threshold on validation.

In [8]:
normal_train = X_train[y_train == 0]
pca_detector = make_pipeline(StandardScaler(), PCA(n_components=0.95, random_state=42))
pca_detector.fit(normal_train)
print('components kept:', pca_detector.named_steps['pca'].n_components_)

record('PCA anomaly detector', pca_detector, reconstruction_error, False,
       'StandardScaler -> PCA(n_components=0.95, random_state=42); fitted on normal training blocks only; '
       'score = squared reconstruction error', len(normal_train))

components kept: 7


threshold 51.9029   cost 1071
accuracy 0.999061   precision 0.999693   recall 0.968230
F1 0.983710   AUC 0.988096   log loss nan


,predicted normal,predicted anomaly
actual normal,111644,1
actual anomaly,107,3261


## 4 · Logistic regression

Counts are standardised so the solver converges and coefficients are comparable.
The first model uses the default equal class weights. The second uses `class_weight='balanced'`, which makes each anomaly count roughly as much as all the normal blocks together during fitting.
Because the threshold is tuned afterwards, both can end up at similar decisions; the weighting mostly moves the scores.

In [9]:
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
logistic.fit(X_train, y_train)
record('Logistic regression', logistic, anomaly_probability, True,
       'StandardScaler -> LogisticRegression(max_iter=1000, random_state=42)', len(y_train))

threshold 0.725094   cost 4
accuracy 0.999965   precision 0.998814   recall 1.000000
F1 0.999407   AUC 1.000000   log loss 0.000414


,predicted normal,predicted anomaly
actual normal,111641,4
actual anomaly,0,3368


In [10]:
logistic_balanced = make_pipeline(
    StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
logistic_balanced.fit(X_train, y_train)
record('Logistic regression balanced', logistic_balanced, anomaly_probability, True,
       "StandardScaler -> LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)",
       len(y_train))

threshold 0.992218   cost 4
accuracy 0.999965   precision 0.998814   recall 1.000000
F1 0.999407   AUC 0.999991   log loss 0.000928


,predicted normal,predicted anomaly
actual normal,111641,4
actual anomaly,0,3368


## 5 · Decision tree and bias-variance

A shallow tree is too simple (high bias: low F1 on both training and validation). A very deep tree memorises the training blocks (high variance: training F1 keeps rising while validation F1 stops improving or falls).
The sweep fits trees of depth 1 to 20 and compares training and validation F1 at the default 0.5 cut. Trees do not need scaled inputs.

In [11]:
sweep = []
for depth in range(1, 21):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    sweep.append({'depth': depth,
                  'train_f1': f1_score(y_train, tree.predict(X_train)),
                  'validation_f1': f1_score(y_val, tree.predict(X_val))})
sweep = pd.DataFrame(sweep)
display(sweep)

,depth,train_f1,validation_f1
0,1,0.551236,0.049410
1,2,0.843474,0.989225
2,3,0.910811,0.995838
3,4,0.969792,0.997774
4,5,0.992826,0.998368
5,6,0.994763,0.998665
6,7,0.996392,0.998665
7,8,0.996640,0.998665
8,9,0.997284,0.998665
9,10,0.997334,0.998665


In [12]:
sweep.to_csv(tables / 'tree_depth_sweep.csv', index=False)

In [13]:
# idxmax returns the first maximum, so among equal scores the shallowest tree wins.
best_depth = int(sweep.loc[sweep['validation_f1'].idxmax(), 'depth'])
print('depth with the highest validation F1:', best_depth)

# Left: the whole sweep. Right: the same lines zoomed in, where the train-validation gap is visible.
fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(11, 4))
for ax in (ax_full, ax_zoom):
    ax.plot(sweep['depth'], sweep['train_f1'], color='black', marker='o', markersize=3)
    ax.plot(sweep['depth'], sweep['validation_f1'], color=ANOMALY_COLOUR, marker='o', markersize=3)
    ax.axvline(best_depth, color=NORMAL_COLOUR, linestyle='--', linewidth=1)
    ax.set_xlabel('Maximum tree depth')
ax_full.set_ylabel('F1 at the default 0.5 cut')
ax_full.set_title('All depths')
ax_zoom.set_xlim(0.5, 23.5)
ax_zoom.set_ylim(0, 1)
ax_zoom.set_title(f'Zoomed: depth 5-20 (chosen depth {best_depth})')
last = sweep.iloc[-1]
ax_zoom.text(last['depth'] + 0.3, last['train_f1'], 'training', va='bottom')
ax_zoom.text(last['depth'] + 0.3, last['validation_f1'], 'validation', va='top', color=ANOMALY_COLOUR)
fig.suptitle('Decision tree depth: bias-variance')
fig.tight_layout()
fig.savefig(figures / 'tree_depth_bias_variance.png', dpi=160)
plt.show()

depth with the highest validation F1: 6


/tmp/ipykernel_11092/3565439717.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
decision_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
decision_tree.fit(X_train, y_train)
record('Decision tree', decision_tree, anomaly_probability, True,
       f'DecisionTreeClassifier(max_depth={best_depth}, random_state=42); depth chosen by validation F1 over 1-20',
       len(y_train))

threshold 0.738095   cost 18
accuracy 0.999922   precision 0.997630   recall 0.999703
F1 0.998665   AUC 0.999702   log loss 0.000646


,predicted normal,predicted anomaly
actual normal,111637,8
actual anomaly,1,3367


### Readable rules

The chosen tree is too deep to read, so a separate **depth-3 tree** is fitted only to explain the main splits.
`export_text` prints each rule; `weights` are the numbers of training blocks [normal, anomaly] that reach each leaf. Counts are whole numbers, so `E11 <= 2.50` means "E11 happened at most 2 times".

In [15]:
rules_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
rules_tree.fit(X_train, y_train)
rules = export_text(rules_tree, feature_names=event_columns, show_weights=True)
print(rules)
(tables / 'decision_tree_rules.txt').write_text(rules)
print('depth-3 tree validation F1 at the 0.5 cut:', round(f1_score(y_val, rules_tree.predict(X_val)), 6))

|--- E26 <= 3.50
|   |--- E9 <= 2.50
|   |   |--- weights: [0.00, 3003.00] class: 1
|   |--- E9 >  2.50
|   |   |--- E20 <= 0.50
|   |   |   |--- weights: [333044.00, 1369.00] class: 0
|   |   |--- E20 >  0.50
|   |   |   |--- weights: [214.00, 1249.00] class: 1
|--- E26 >  3.50
|   |--- E23 <= 1.00
|   |   |--- E26 <= 10.00
|   |   |   |--- weights: [1550.00, 21.00] class: 0
|   |   |--- E26 >  10.00
|   |   |   |--- weights: [1.00, 5.00] class: 1
|   |--- E23 >  1.00
|   |   |--- E2 <= 3.50
|   |   |   |--- weights: [100.00, 4454.00] class: 1
|   |   |--- E2 >  3.50
|   |   |   |--- weights: [24.00, 1.00] class: 0



depth-3 tree validation F1 at the 0.5 cut: 0.995838


In [16]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(rules_tree, feature_names=event_columns, class_names=['normal', 'anomaly'],
          impurity=False, proportion=False, rounded=True, fontsize=9, ax=ax)
ax.set_title('Depth-3 decision tree used for the readable rules')
fig.tight_layout()
fig.savefig(figures / 'decision_tree.png', dpi=160)
plt.show()

/tmp/ipykernel_11092/2407910943.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · k-nearest neighbours

kNN measures distance, so counts are standardised first; otherwise a large count like `E3` would swamp everything else.
The full training part is used (no sample): identical validation vectors are scored once and mapped back to all blocks.

The kNN score is the share of the k nearest training blocks that are anomalies, so it can only take k + 1 values (0, 1/k, …, 1). **A larger k gives more possible distinct score values**, which gives the threshold more room; the `distinct_scores` column shows how many actually occur on validation. k = 5, 15 and 25 are compared with the same cost rule, and the cheapest k is kept.

In [17]:
knn_rows = []
knn_models = {}
for k in [5, 15, 25]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    knn.fit(X_train, y_train)
    scores = knn_probability(knn, X_val)
    threshold, cost = choose_threshold(y_val, scores)
    knn_rows.append({'k': k, 'distinct_scores': len(np.unique(scores)),
                     'threshold': threshold, 'validation_cost': cost})
    knn_models[k] = knn
knn_table = pd.DataFrame(knn_rows)
display(knn_table)

,k,distinct_scores,threshold,validation_cost
0,5,3,0.60,21
1,15,6,0.60,19
2,25,7,0.04,12


In [18]:
# idxmin returns the first minimum, so a tie goes to the smaller k.
best_k = int(knn_table.loc[knn_table['validation_cost'].idxmin(), 'k'])
print('chosen k:', best_k)
record('kNN', knn_models[best_k], knn_probability, True,
       f'StandardScaler -> KNeighborsClassifier(n_neighbors={best_k}); k chosen from 5, 15, 25 by validation cost; '
       'full training part, no sample', len(y_train))

chosen k: 25


threshold 0.04   cost 12
accuracy 0.999896   precision 0.996450   recall 1.000000
F1 0.998222   AUC 1.000000   log loss 0.000212


,predicted normal,predicted anomaly
actual normal,111633,12
actual anomaly,0,3368


## 7 · Gaussian Naive Bayes

Naive Bayes assumes the events are independent given the class. They are not (several events are near-perfect twins, see notebook 02), so the model counts the same evidence several times and its probabilities are pushed to the extremes: many blocks get exactly 1.0 and most others sit extremely close to 0.
Its ranking can still be good (high AUC); the threshold search over its own distinct scores finds a usable cut.

In [19]:
naive_bayes = GaussianNB()
naive_bayes.fit(X_train, y_train)
record('Naive Bayes', naive_bayes, anomaly_probability, True, 'GaussianNB() with default var_smoothing=1e-9',
       len(y_train))

threshold 1   cost 109
accuracy 0.999052   precision 0.968651   recall 1.000000
F1 0.984076   AUC 0.999512   log loss 0.034159


,predicted normal,predicted anomaly
actual normal,111536,109
actual anomaly,0,3368


In [20]:
nb_scores = validation_scores['Naive Bayes']
print('distinct validation scores:', len(np.unique(nb_scores)))
print('share of scores below 0.000001:', round(np.mean(nb_scores < 1e-6), 4))
print('share of scores exactly 1:', round(np.mean(nb_scores == 1), 4))

# The old bug, reproduced: the same cut of 1.0 with the two different comparisons.
print('blocks flagged by score >= 1.0:', np.sum(nb_scores >= 1.0))
print('blocks flagged by score >  1.0:', np.sum(nb_scores > 1.0))

distinct validation scores: 16
share of scores below 0.000001: 0.9698
share of scores exactly 1: 0.0302
blocks flagged by score >= 1.0: 3477
blocks flagged by score >  1.0: 0


## 8 · Linear SVM

`LinearSVC` finds a straight boundary between the classes. Its score is the signed distance from that boundary, not a probability, so log loss is left empty.
It is fast enough to use the full training part. `dual=False` is the recommended solver when there are many more blocks than features.

In [21]:
svm = make_pipeline(StandardScaler(), LinearSVC(dual=False, max_iter=5000, random_state=42))
svm.fit(X_train, y_train)
record('Linear SVM', svm, svm_margin, False,
       'StandardScaler -> LinearSVC(dual=False, max_iter=5000, random_state=42); score = decision_function',
       len(y_train))

threshold -0.952167   cost 12
accuracy 0.999896   precision 0.996450   recall 1.000000
F1 0.998222   AUC 0.999991   log loss nan


,predicted normal,predicted anomaly
actual normal,111633,12
actual anomaly,0,3368


## 9 · Random forest (bagging)

A random forest trains many deep trees, each on a bootstrap sample of the training blocks and a random subset of events at each split, then averages them. Averaging cuts the variance that a single deep tree has. Its score is the average anomaly probability over the 200 trees.

In [22]:
forest = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
forest.fit(X_train, y_train)
record('Random forest', forest, anomaly_probability, True,
       "RandomForestClassifier(n_estimators=200, max_features='sqrt' (default), max_depth=None (default), "
       'random_state=42, n_jobs=-1)', len(y_train))

threshold 0.73   cost 32
accuracy 0.999957   precision 0.999406   recall 0.999109
F1 0.999258   AUC 0.999231   log loss 0.540273


,predicted normal,predicted anomaly
actual normal,111643,2
actual anomaly,3,3365


In [23]:
importance = pd.DataFrame({'event': event_columns, 'importance': forest.feature_importances_})
importance = importance.sort_values('importance', ascending=False)
importance.to_csv(tables / 'event_importance.csv', index=False)
display(importance.head(10))

,event,importance
24,E26,0.153235
20,E21,0.114868
19,E20,0.103288
25,E27,0.086404
10,E11,0.084707
8,E9,0.083583
4,E5,0.073126
26,E28,0.052623
21,E23,0.043515
12,E13,0.040881


## 10 · AdaBoost (boosting)

AdaBoost trains decision stumps (one-split trees) one after another; each new stump focuses on the blocks the previous ones got wrong. Where the forest reduces variance by averaging, boosting reduces bias by adding.

In [24]:
adaboost = AdaBoostClassifier(n_estimators=200, random_state=42)
adaboost.fit(X_train, y_train)
record('AdaBoost', adaboost, anomaly_probability, True,
       'AdaBoostClassifier(n_estimators=200, estimator=depth-1 stump (default), learning_rate=1.0, random_state=42)',
       len(y_train))

threshold 0.489054   cost 12
accuracy 0.999896   precision 0.996450   recall 1.000000
F1 0.998222   AUC 0.999997   log loss 0.558342


,predicted normal,predicted anomaly
actual normal,111633,12
actual anomaly,0,3368


AdaBoost's `predict_proba` is not a calibrated probability: its scores stay close to 0.5, so its log loss is poor even when its ranking (AUC) is good. The range below shows this.

In [25]:
ada_scores = validation_scores['AdaBoost']
print('lowest AdaBoost score:', round(ada_scores.min(), 4), '  highest:', round(ada_scores.max(), 4))

lowest AdaBoost score: 0.4157   highest: 0.6446


## 11 · Small MLP

A neural network with two hidden layers of 32 and 16 units. Inputs are standardised. `early_stopping=True` holds back 10% of the training part and stops when its score stops improving, which guards against overfitting.

In [26]:
mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32, 16), early_stopping=True,
                                                    max_iter=100, random_state=42))
mlp.fit(X_train, y_train)
print('training epochs run:', mlp.named_steps['mlpclassifier'].n_iter_)
record('MLP', mlp, anomaly_probability, True,
       'StandardScaler -> MLPClassifier(hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=100, '
       'random_state=42)', len(y_train))

training epochs run: 12
threshold 0.780178   cost 3
accuracy 0.999974   precision 0.999110   recall 1.000000
F1 0.999555   AUC 1.000000   log loss 0.000103


,predicted normal,predicted anomaly
actual normal,111642,3
actual anomaly,0,3368


## 12 · Validation comparison

All models are sorted by validation cost. Only the four primary classifiers compete for final selection; other models are supplementary.

In [27]:
columns = ['model', 'threshold', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'log_loss',
           'tn', 'fp', 'fn', 'tp', 'validation_cost']
comparison = pd.DataFrame(results).rename(columns={'cost': 'validation_cost'})
comparison = comparison.sort_values(['validation_cost', 'f1'], ascending=[True, False])[columns]
comparison = comparison.reset_index(drop=True)
comparison.to_csv(tables / 'model_comparison.csv', index=False)
display(comparison)

,model,threshold,accuracy,precision,recall,f1,auc,log_loss,tn,fp,fn,tp,validation_cost
0,MLP,0.780178,0.999974,0.999110,1.000000,0.999555,1.000000,0.000103,111642,3,0,3368,3
1,Logistic regression,0.725094,0.999965,0.998814,1.000000,0.999407,1.000000,0.000414,111641,4,0,3368,4
2,Logistic regression balanced,0.992218,0.999965,0.998814,1.000000,0.999407,0.999991,0.000928,111641,4,0,3368,4
3,kNN,0.040000,0.999896,0.996450,1.000000,0.998222,1.000000,0.000212,111633,12,0,3368,12
4,Linear SVM,-0.952167,0.999896,0.996450,1.000000,0.998222,0.999991,NaN,111633,12,0,3368,12
5,AdaBoost,0.489054,0.999896,0.996450,1.000000,0.998222,0.999997,0.558342,111633,12,0,3368,12
6,Decision tree,0.738095,0.999922,0.997630,0.999703,0.998665,0.999702,0.000646,111637,8,1,3367,18
7,Random forest,0.730000,0.999957,0.999406,0.999109,0.999258,0.999231,0.540273,111643,2,3,3365,32
8,Naive Bayes,1.000000,0.999052,0.968651,1.000000,0.984076,0.999512,0.034159,111536,109,0,3368,109
9,PCA anomaly detector,51.902880,0.999061,0.999693,0.968230,0.983710,0.988096,NaN,111644,1,107,3261,1071


In [28]:
# Save every model's validation scores, so the website can draw ROC and precision-recall
# curves for all models without training anything again.
score_names = list(validation_scores)
np.savez_compressed(processed / 'validation_scores.npz', model_names=np.array(score_names),
                    scores=np.vstack([validation_scores[name] for name in score_names]))
print('saved validation scores for', len(score_names), 'models')

saved validation scores for 11 models


In [29]:
model_settings = pd.DataFrame(settings)
model_settings.to_csv(tables / 'model_settings.csv', index=False)
pd.set_option('display.max_colwidth', None)
display(model_settings)

,model,settings,training_rows,scikit_learn_version
0,Baseline (always normal),DummyClassifier(strategy='prior'),345035,1.9.1
1,PCA anomaly detector,"StandardScaler -> PCA(n_components=0.95, random_state=42); fitted on normal training blocks only; score = squared reconstruction error",334933,1.9.1
2,Logistic regression,"StandardScaler -> LogisticRegression(max_iter=1000, random_state=42)",345035,1.9.1
3,Logistic regression balanced,"StandardScaler -> LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)",345035,1.9.1
4,Decision tree,"DecisionTreeClassifier(max_depth=6, random_state=42); depth chosen by validation F1 over 1-20",345035,1.9.1
5,kNN,"StandardScaler -> KNeighborsClassifier(n_neighbors=25); k chosen from 5, 15, 25 by validation cost; full training part, no sample",345035,1.9.1
6,Naive Bayes,GaussianNB() with default var_smoothing=1e-9,345035,1.9.1
7,Linear SVM,"StandardScaler -> LinearSVC(dual=False, max_iter=5000, random_state=42); score = decision_function",345035,1.9.1
8,Random forest,"RandomForestClassifier(n_estimators=200, max_features='sqrt' (default), max_depth=None (default), random_state=42, n_jobs=-1)",345035,1.9.1
9,AdaBoost,"AdaBoostClassifier(n_estimators=200, estimator=depth-1 stump (default), learning_rate=1.0, random_state=42)",345035,1.9.1


Precision against recall on validation for the three cheapest models. The dot marks each model's chosen threshold. The axes are shown across the full range.

In [30]:
primary_models = ['Logistic regression', 'Decision tree', 'kNN', 'Random forest']
top_three = comparison.loc[comparison['model'].isin(primary_models), 'model'].head(3).tolist()
line_styles = ['-', '--', ':']

fig, ax = plt.subplots(figsize=(7, 5))
for name, style in zip(top_three, line_styles):
    precision, recall, _ = precision_recall_curve(y_val, validation_scores[name])
    ax.plot(recall, precision, color='black', linestyle=style, label=name)
    chosen = comparison.set_index('model').loc[name]
    ax.plot(chosen['recall'], chosen['precision'], 'o', color=ANOMALY_COLOUR)
ax.set_xlim(0, 1.0005)
ax.set_ylim(0, 1.001)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Validation precision-recall, top three (dot = chosen threshold)')
ax.legend(frameon=False, loc='lower left')
fig.tight_layout()
fig.savefig(figures / 'validation_precision_recall.png', dpi=160)
plt.show()

/tmp/ipykernel_11092/1531406983.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13 · Test set, used once

The three cheapest primary classifiers are scored on the test blocks with the thresholds chosen on validation. Nothing is re-tuned after this.

In [31]:
# Count how many times the test set is scored, for the leakage check below.
test_set_uses = 0

test_rows = []
test_scores = {}
test_set_uses += 1
for name in top_three:
    model, score_function, gives_probability = fitted[name]
    threshold = comparison.set_index('model').loc[name, 'threshold']
    test_scores[name] = score_function(model, X_test)
    row = evaluate(y_test, test_scores[name], threshold, gives_probability)
    row['model'] = name
    test_rows.append(row)
    print(name)
    show(row)

Logistic regression
threshold 0.725094   cost 221
accuracy 0.999800   precision 0.999701   recall 0.993468
F1 0.996575   AUC 0.999398   log loss 0.001149


,predicted normal,predicted anomaly
actual normal,111644,1
actual anomaly,22,3346


kNN
threshold 0.04   cost 25
accuracy 0.999939   precision 0.998517   recall 0.999406
F1 0.998961   AUC 0.999703   log loss 0.000950


,predicted normal,predicted anomaly
actual normal,111640,5
actual anomaly,2,3366


Decision tree
threshold 0.738095   cost 71
accuracy 0.999930   precision 0.999703   recall 0.997922
F1 0.998811   AUC 0.999550   log loss 0.000971


,predicted normal,predicted anomaly
actual normal,111644,1
actual anomaly,7,3361


In [32]:
final_columns = ['model', 'threshold', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'log_loss',
                 'tn', 'fp', 'fn', 'tp', 'test_cost']
final = pd.DataFrame(test_rows).rename(columns={'cost': 'test_cost'})[final_columns]
final.to_csv(tables / 'final_test_results.csv', index=False)
display(final)

,model,threshold,accuracy,precision,recall,f1,auc,log_loss,tn,fp,fn,tp,test_cost
0,Logistic regression,0.725094,0.999800,0.999701,0.993468,0.996575,0.999398,0.001149,111644,1,22,3346,221
1,kNN,0.040000,0.999939,0.998517,0.999406,0.998961,0.999703,0.000950,111640,5,2,3366,25
2,Decision tree,0.738095,0.999930,0.999703,0.997922,0.998811,0.999550,0.000971,111644,1,7,3361,71


In [33]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, style in zip(top_three, line_styles):
    false_positive_rate, true_positive_rate, _ = roc_curve(y_test, test_scores[name])
    auc = final.set_index('model').loc[name, 'auc']
    ax.plot(false_positive_rate, true_positive_rate, color='black', linestyle=style,
            label=f'{name} (AUC {auc:.6f})')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.0005)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('Test ROC curves, top three validation models')
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
fig.savefig(figures / 'test_roc_curves.png', dpi=160)
plt.show()

/tmp/ipykernel_11092/4216968861.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Leakage checks

Only event counts are model features; E22 is removed because it is constant in training. The partitions share no rows or event-count vectors. Scalers, PCA and models are fitted on training rows only, and selection uses validation data.


In [34]:
print('features used:', event_columns)
assert event_columns == [f'E{number}' for number in range(1, 30) if number != 22]
assert df['E22'].iloc[splits['train_idx']].nunique() == 1
for column in ['BlockId', 'Label', 'Type', 'target', 'E22']:
    assert column not in event_columns
for left, right in [('train_idx', 'validation_idx'), ('train_idx', 'test_idx'), ('validation_idx', 'test_idx')]:
    left_patterns = pd.MultiIndex.from_frame(df.iloc[splits[left]][event_columns])
    right_patterns = pd.MultiIndex.from_frame(df.iloc[splits[right]][event_columns])
    assert not left_patterns.isin(right_patterns).any(), (left, right)
print('Feature order, constant E22 and zero cross-split pattern overlap: OK')


features used: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E23', 'E24', 'E25', 'E26', 'E27', 'E28', 'E29']


Feature order, constant E22 and zero cross-split pattern overlap: OK


In [35]:
def training_rows_seen(model):
    # Each kind of model stores the number of rows it was fitted on in a different place.
    if hasattr(model, 'named_steps'):
        # A pipeline: the scaler is fitted first and counts every row passed to fit.
        return int(model.named_steps['standardscaler'].n_samples_seen_)
    if isinstance(model, RandomForestClassifier):
        # Bootstrap draws are stored as weights; they add up to the number of training rows.
        return int(model.estimators_[0].tree_.weighted_n_node_samples[0])
    if isinstance(model, AdaBoostClassifier):
        return int(model.estimators_[0].tree_.n_node_samples[0])
    if isinstance(model, DecisionTreeClassifier):
        return int(model.tree_.n_node_samples[0])
    if isinstance(model, GaussianNB):
        return int(model.class_count_.sum())
    return None

In [36]:
expected_rows = model_settings.set_index('model')['training_rows']
row_check = []
for name, (model, _, _) in fitted.items():
    if name == 'Baseline (always normal)':
        continue
    row_check.append({'model': name, 'rows_seen_in_fit': training_rows_seen(model),
                      'expected_training_rows': expected_rows[name]})
row_check = pd.DataFrame(row_check)
row_check['match'] = row_check['rows_seen_in_fit'] == row_check['expected_training_rows']
display(row_check)
assert row_check['match'].all()

,model,rows_seen_in_fit,expected_training_rows,match
0,PCA anomaly detector,334933,334933,True
1,Logistic regression,345035,345035,True
2,Logistic regression balanced,345035,345035,True
3,Decision tree,345035,345035,True
4,kNN,345035,345035,True
5,Naive Bayes,345035,345035,True
6,Linear SVM,345035,345035,True
7,Random forest,345035,345035,True
8,AdaBoost,345035,345035,True
9,MLP,345035,345035,True


In [37]:
pca_rows = pca_detector.named_steps['pca'].n_samples_
print('PCA fitted on', pca_rows, 'rows; normal training blocks:', len(normal_train), '; match:', pca_rows == len(normal_train))
print('baseline score equals the training anomaly rate:', np.isclose(baseline.class_prior_[1], y_train.mean()))
print('training rows:', len(y_train), ' validation rows:', len(y_val), ' test rows:', len(y_test))
print('test rows shared with training or validation:',
      len(np.intersect1d(splits['test_idx'], np.concatenate([splits['train_idx'], splits['validation_idx']]))))
print('times the test set was scored:', test_set_uses)
assert pca_rows == len(normal_train) and test_set_uses == 1

PCA fitted on 334933 rows; normal training blocks: 334933 ; match: True
baseline score equals the training anomaly rate: True
training rows: 345035  validation rows: 115013  test rows: 115013
test rows shared with training or validation: 0
times the test set was scored: 1


## 15 · Save the chosen model

The primary classifier with the lowest **validation** cost is saved with its threshold, so the website and any later script use exactly the same decision rule. It was chosen before the test set was scored.

In [38]:
best_name = top_three[0]
best_model, best_score_function, _ = fitted[best_name]
package = {
    'model_name': best_name,
    'model': best_model,
    'threshold': float(comparison.set_index('model').loc[best_name, 'threshold']),
    'event_columns': event_columns,
    'decision_rule': 'alarm when predict_proba(X)[:, 1] >= threshold',
    'scikit_learn_version': sklearn.__version__,
}
with open(models_folder / 'best_final_model.pkl', 'wb') as file:
    pickle.dump(package, file)
print('saved', best_name, 'with threshold', package['threshold'])

saved Logistic regression with threshold 0.725093889698804


## Conclusion and limitations

The printed tables above are the results; the cell below repeats the headline test numbers for the chosen model.

- The threshold rule (10 × misses + false alarms) is an assumption. A different cost ratio would move every threshold.
- This is one labelled benchmark from one Hadoop deployment. The same model or threshold may not carry over to another system.
- Event counts show patterns associated with known labels; they do not explain why a block failed.

In [39]:
chosen = final.set_index('model').loc[best_name]
print('chosen model:', best_name)
print('test precision', round(chosen['precision'], 6), ' recall', round(chosen['recall'], 6),
      ' F1', round(chosen['f1'], 6), ' AUC', round(chosen['auc'], 6))
print('test missed anomalies (FN):', int(chosen['fn']), '  false alarms (FP):', int(chosen['fp']),
      '  out of', len(y_test), 'test blocks')

chosen model: Logistic regression
test precision 0.999701  recall 0.993468  F1 0.996575  AUC 0.999398
test missed anomalies (FN): 22   false alarms (FP): 1   out of 115013 test blocks
